# Milestone 1.1 & 1.2: Dual Storage Engine & Medallion OHLCV Pipeline

## 1. What We Built
- **`SQLiteLedger` (`src/stockcortex/storage/sqlite_ledger.py`)**: Configured SQLite in Write-Ahead Logging (`WAL`) mode with foreign keys enabled, housing `prediction_ledger`, `ingestion_audit_log`, and `portfolios`.
- **`DuckStore` (`src/stockcortex/storage/duck_store.py`)**: Built a hardware-calibrated DuckDB Parquet engine capped at `6GB` RAM and `4` CPU threads.
- **`MarketDataLoader` (`src/stockcortex/ingestion/market_loader.py`)**: Implemented the `.NS -> .BO -> Alpha Vantage` fallback chain, immutable UTC-stamped Bronze snapshots (`data/bronze/`), and Silver OHLC invariant enforcement (`data/silver/`).

## 2. Why These Architectural Decisions Were Made
| Design Choice | Engineering Rationale |
| :--- | :--- |
| **Hybrid SQLite WAL + DuckDB Parquet** | SQLite excels at concurrent, ACID-compliant row appends (`prediction_ledger`) without file locks, while DuckDB scans columnar Parquet files orders of magnitude faster for multi-year time-series features. |
| **Immutable Bronze Snapshots** | Persisting raw vendor responses before cleaning guarantees full reproducibility and lets us re-run Silver/Gold transformations without re-querying rate-limited APIs. |
| **Strict OHLC Invariant Filtering** | Free market feeds occasionally emit zero-volume placeholders or inverted High/Low ticks; quarantining them in Silver prevents `NaN`/`Inf` explosions in log-return and Garman-Klass volatility features. |

## 3. How to Run Live NIFTY Starter Ingestion & Audit Results

In [1]:
from stockcortex.ingestion.market_loader import MarketDataLoader

loader = MarketDataLoader()
outputs = loader.ingest_universe(period="5y")
print(f"Ingested {len(outputs)} starter NIFTY symbols into Silver Parquet:")
print(list(outputs.keys()))

Ingested 10 starter NIFTY symbols into Silver Parquet:
['RELIANCE', 'TCS', 'HDFCBANK', 'INFY', 'ICICIBANK', 'BHARTIARTL', 'SBIN', 'LT', 'ITC', 'HINDUNILVR']


In [2]:
from stockcortex.config import settings
from stockcortex.storage.duck_store import DuckStore

duck = DuckStore()
silver_glob = (settings.silver_dir / "*_ohlcv.parquet").as_posix()

summary = duck.query_parquet(f"""
    SELECT symbol, source_ticker, COUNT(*) AS trading_days,
           MIN(date) AS first_date, MAX(date) AS last_date,
           ROUND(AVG(close), 2) AS avg_close
    FROM read_parquet('{silver_glob}')
    GROUP BY symbol, source_ticker
    ORDER BY symbol
""")
summary

,symbol,source_ticker,trading_days,first_date,last_date,avg_close
0,BHARTIARTL,BHARTIARTL.NS,1236,2021-11-26,2026-11-26,1307.17
1,HDFCBANK,HDFCBANK.NS,1236,2021-11-26,2026-11-26,787.79
2,HINDUNILVR,HINDUNILVR.NS,1236,2021-11-26,2026-11-26,2286.01
3,ICICIBANK,ICICIBANK.NS,1236,2021-11-26,2026-11-26,1094.22
4,INFY,INFY.NS,1236,2021-11-26,2026-11-26,1382.66
5,ITC,ITC.NS,1236,2021-11-26,2026-11-26,333.55
6,LT,LT.NS,1236,2021-11-26,2026-11-26,3004.49
7,RELIANCE,RELIANCE.NS,1236,2021-11-26,2026-11-26,1277.36
8,SBIN,SBIN.NS,1236,2021-11-26,2026-11-26,711.42
9,TCS,TCS.NS,1236,2021-11-26,2026-11-26,3097.97
